In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time
import os
import requests

def scrape_health_kr_final(query, download_path='official_pill_images'):
    os.makedirs(download_path, exist_ok=True)

    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    wait = WebDriverWait(driver, 10)

    try:
        print(f"\n[{query}] 약학정보원 접속 중...")
        driver.get("https://www.health.kr/")
        time.sleep(2)

        # 팝업 닫기
        try:
            alert = driver.switch_to.alert
            alert.accept()
        except:
            pass

        # 1. 검색어 입력
        search_box = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input[id='search_word'], input[name='search_word']"))
        )
        search_box.clear()
        search_box.send_keys(query)
        time.sleep(0.5)
        search_box.send_keys(Keys.RETURN)

        print("검색 완료. 상세 페이지 진입 시도 중...")
        time.sleep(3) 

        # 2. 첫 번째 검색 결과 클릭
        try:
            first_result_link = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "table tbody tr td a"))
            )
            first_result_link.click()
        except Exception:
            print(f"❌ '{query}'에 대한 검색 결과가 없습니다.")
            return

        time.sleep(3)

        # 3. 새 창으로 포커스 이동
        if len(driver.window_handles) > 1:
            driver.switch_to.window(driver.window_handles[1])
            time.sleep(2)

        print("상세 페이지 진입 완료! 이미지 주소 훔쳐오는 중...")
        
        # 4. 페이지 안의 모든 이미지 태그 검색
        img_elements = driver.find_elements(By.TAG_NAME, 'img')
        target_img_url = None

        for img in img_elements:
            src = img.get_attribute('src')
            # 캡처해주신 약학정보원의 고유 이미지 폴더 'sb_photo'가 포함된 이미지 찾기
            if src and 'common.health.kr/shared/images/sb_photo/' in src:
                target_img_url = src
                break # 찾았으면 즉시 중단

        if target_img_url:
            # 5. 핵심 꼼수: 파일명(예: A11ABBBB228101.jpg)만 쏙 빼내서 고화질 주소로 강제 조립!
            file_name_only = target_img_url.split('/')[-1]
            high_res_url = f"https://common.health.kr/shared/images/sb_photo/big3/{file_name_only}"
            
            print(f"💡 고화질 강제 변환 주소: {high_res_url}")
            
            # 6. 다운로드
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(high_res_url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                file_name = os.path.join(download_path, f"{query}_고화질.jpg")
                with open(file_name, 'wb') as f:
                    f.write(response.content)
                print(f"✅ 사진 저장 완료: {file_name}")
            else:
                print(f"❌ 다운로드 실패 (상태 코드: {response.status_code})")
        else:
            print(f"❌ 화면에서 '{query}'의 사진(sb_photo)을 찾지 못했습니다.")

    except Exception as e:
        print(f"🚨 에러 발생: {e}")
    
    finally:
        driver.quit()

if __name__ == '__main__':
    pill_list = ["자이로릭정", "보나링에이정", "후릭스정", "리단정", "마그밀정", "구주스피로닥톤정"]
    for pill in pill_list:
        scrape_health_kr_final(pill)
        time.sleep(2)


[자이로릭정] 약학정보원 접속 중...
검색 완료. 상세 페이지 진입 시도 중...
상세 페이지 진입 완료! 이미지 주소 훔쳐오는 중...
💡 고화질 강제 변환 주소: https://common.health.kr/shared/images/sb_photo/big3/A11A0500A006901.jpg
✅ 사진 저장 완료: official_pill_images\자이로릭정_고화질.jpg

[보나링에이정] 약학정보원 접속 중...
검색 완료. 상세 페이지 진입 시도 중...
상세 페이지 진입 완료! 이미지 주소 훔쳐오는 중...
💡 고화질 강제 변환 주소: https://common.health.kr/shared/images/sb_photo/big3/A11A0950A004302.jpg
✅ 사진 저장 완료: official_pill_images\보나링에이정_고화질.jpg

[후릭스정] 약학정보원 접속 중...
검색 완료. 상세 페이지 진입 시도 중...
상세 페이지 진입 완료! 이미지 주소 훔쳐오는 중...
💡 고화질 강제 변환 주소: https://common.health.kr/shared/images/sb_photo/big3/A11A0950A006801.jpg
✅ 사진 저장 완료: official_pill_images\후릭스정_고화질.jpg

[리단정] 약학정보원 접속 중...
검색 완료. 상세 페이지 진입 시도 중...
상세 페이지 진입 완료! 이미지 주소 훔쳐오는 중...
💡 고화질 강제 변환 주소: https://common.health.kr/shared/images/sb_photo/big3/A11A1310A002602.jpg
✅ 사진 저장 완료: official_pill_images\리단정_고화질.jpg

[마그밀정] 약학정보원 접속 중...
검색 완료. 상세 페이지 진입 시도 중...
상세 페이지 진입 완료! 이미지 주소 훔쳐오는 중...
💡 고화질 강제 변환 주소: https://common.health.kr/shared/images/sb_photo/bi